In [ ]:
#%%time

import numpy as np
import h5py
import re
import pickle

# Open ARCHS4 v2.1 Data 
infile = "/home/ajoshi/archs4/data/archs4_gene_human_v2.1.h5"

# - Use read_direct method to improve read performance
with h5py.File(infile, 'r') as hdf5_file:
    data = hdf5_file['data']
    expression = data['expression'][:]

    meta = hdf5_file['meta']
    symbols = meta['genes']['gene_symbol'][:]
    source_name_ch1 = meta['samples']['source_name_ch1'][:]

# Decode source names using list comprehension
source_decoded = [x.decode('latin-1') for x in source_name_ch1]

# Clean source names using regular expressions
source_cleaned = [re.sub(r'[-/\'\\ _]', '', x) for x in source_decoded]

# List of all categories (keywords)
keyword_list = [# Cardiovascular 
                "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle",
                "HUVEC", # Endothelial 
                
                # Connective Tissue
                "Adipose", "Adipocyte",
                "Myofibroblast", "Osteoblast", "Stromal Cell", "MG63", "K562", # Bone
                "Bone Marrow", 
                "Chondrocyte", 
                "HT1080", "NHDF", # Connective
                
                # Digestive
                "Esophagus",
                "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", "Small Intestine", 
                "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell", "HEPG2", # Liver
                "Alpha Cell", "Beta Cell", "Pancreatic Islet", "PANC1", # Pancreas
                "Gastric Epithelial Cell", 
                "HCT116", "HT29", "RKO", # Connective, Colon
                
                # Immune
                "Granulocyte", "Neutrophil",
                "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", "Tlymphocyte", 
                "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", # Lymphoid
                "Alveolar", "Dendritic", "Kupffer", "Macrophage", "Macroglia", 
                "THP1", "U937", # Myeloid
                "JEG3", # Placenta
                "Spleen", 
                "Thymus", "Thymocyte", 
                
                # Integumentary
                "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte", 
                "A431", "HACAT", "HNSCC",
                
                # Muscular
                "Skeletal Muscle", "Myoblast", 
                "Airway Smooth Muscle", #"Myofibroblast", [Repeat]
                "Respiratory Smooth Muscle", "Vascular Smooth Muscle", 
                "LHCNM2", # Muscle
                
                # Nervous
                "Astrocyte", "Cerebellum", "Cerebral Cortex", "Hypothalamus", "Medula", 
                "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", "Thalamus", 
                "Retina", 
                "Motor Neuron", "Sensory Neuron",
                "Brain", # Example in the ARCHS4 Website [1]
                "DAOY", "IMR32", "SKNSH", # Brain 
                
                # Respiratory 
                #"Alveolar", [Repeat] 
                "Lung", "Lung Epithelial", 
                "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", # Lung
                "Trachea", 
                
                # Urogenital 
                "Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D",
                "HELA", # Cervix
                "Kidney", "Podocyte", "Renal Cortex",
                "293F", "FLPIN TREX 293", "HEK293", # Kidne
                "C42", "DU145", "LNCAP", "PC3", # Prosta
                
                "Ovary", "Granulosa", "Oocyte",
                "ES2", "SKOV3", # Ovary
                "Testis", 
                "GSE81547", # Example in the ARCHS4 GitHub
                "GSE30017",  "GSM2679484"] # Examples in the ARCHS4 Website [2]

# Create a dictionary that maps keywords to their indices
category_dict = {keyword: [idx for idx, string in enumerate(source_cleaned) 
                           if keyword.replace(" ", "").lower() in string.lower()] for keyword in keyword_list}

# Get the total sample size collected for each keyword 
category_sample_sizes = {key: len(category_dict[key]) for key in category_dict}

# The 57k matched genes between the Gene-Length-File & g:Convert-csv

import pickle

pickle_in = open("/home/lwu/ARCHS4_v2-1/Data Results/matched_genes_bt_gConvertCsv_GLF.pkl", 'rb')
csv_glf_genes_dict = pickle.load(pickle_in)
pickle_in.close()

print("# Matched Genes:", len(csv_glf_genes_dict))

# print()
# print("Check First 3 Genes in CSV-GLF Dict:")
# for key, value in islice(csv_glf_genes_dict.items(), 3):
#     print(key, value)
    
#%%time
num_rows, num_cols = expression.shape
new_data = []
for gene_row in range(num_rows):
    
    # Gene index
    #print(gene_row)
    
    # Gene name 
    gene_name = list(csv_glf_genes_dict.keys())[gene_row]
    #prinzt("gene_name:", gene_name)
    
    # Gene data
    gene_data = expression[gene_row]
    #print("row:", gene_data[0:5])
        
    # GLF Gene name 
    glf_gene_name = csv_glf_genes_dict[gene_name]
    #print("glf gene_name:", glf_gene_name)
    
    # Gene length 
    gene_length = wm_dict[glf_gene_name]
    #print("length", gene_length)
    
    #print(type(gene_data))
    #print(type(gene_length))
    # New row 
    new_gene_row = gene_data/int(gene_length)
    #print("new_row:", new_gene_row[0:5])
    test = new_gene_row.tolist()
    new_data.append(test)
    #break
pickle_out = open("corrected_selectedSamples_readCounts_matrix.pkl", 'wb')
pickle.dump(new_data, pickle_out)
pickle_out.close()